# Q2. Supplier geography: countries, US states, overseas vs domestic

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** How are Net PO value and PO item volume distributed across supplier countries and US
states, and what share relies on overseas vs domestic suppliers?

Geography = supplier master **address** (LFA1 country and region). It isn't the ship-from location.

In [ ]:
w["origin"] = np.select([w["Country/Region Key"].isna(), w["Country/Region Key"] == "US"],
                        ["Unknown", "Domestic (US)"], "Overseas")
geo = (w.groupby("origin").agg(net=("Net Order Value", "sum"), items=("Item", "size"),
                               suppliers=("Supplier", "nunique"))
         .assign(value_share=lambda d: d.net / d.net.sum(), item_share=lambda d: d["items"] / d["items"].sum())
         .sort_values("net", ascending=False))
geo.style.format({"net": "{:,.0f}", "value_share": "{:.1%}", "item_share": "{:.1%}"})

In [ ]:
order = ["Domestic (US)", "Overseas", "Unknown"]
g = geo.reindex(order)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), sharey=True)
for ax, col, title in [(axes[0], "value_share", "Share of Net PO value"), (axes[1], "item_share", "Share of PO items")]:
    ax.barh(order[::-1], g[col][::-1], color=[MUTED, SERIES[1], SERIES[0]], height=0.6)
    for y, v in enumerate(g[col][::-1]):
        ax.text(v, y, f"  {v:.1%}", va="center", color=INK_2)
    ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 1.15); ax.set_xticks([0, .25, .5, .75, 1]); ax.set_title(title)
    ax.grid(axis="y", visible=False)
fig.suptitle(f"Domestic vs overseas suppliers, US00, {PERIOD}", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "q2_domestic_overseas")

In [ ]:
by_country = (w.groupby(w["Country/Region Key"].fillna("Unknown"))
                .agg(net=("Net Order Value", "sum"), items=("Item", "size"), suppliers=("Supplier", "nunique"))
                .assign(share=lambda d: d.net / d.net.sum())
                .sort_values("net", ascending=False))
display(by_country.style.format({"net": "{:,.0f}", "share": "{:.2%}"}))

c = by_country[by_country["net"] > 0].iloc[::-1]   # log axis can't show 0
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(c.index, c["net"], color=[SERIES[0] if i == "US" else SERIES[1] for i in c.index], height=0.65)
for y, (v, n) in enumerate(zip(c["net"], c["items"])):
    ax.text(v, y, f"  ${v:,.0f} · {n} items", va="center", color=INK_2, fontsize=8.5)
ax.set_xscale("log"); ax.xaxis.set_major_formatter(usd); ax.set_xlim(right=c["net"].max() * 30)
ax.grid(axis="y", visible=False)
ax.set(title="Net PO value by supplier country (log scale)", xlabel="Net PO value, USD (log scale)")
save(fig, "q2_by_country")

In [ ]:
us = w[w["Country/Region Key"] == "US"].assign(state=lambda d: d["Region"].fillna("Unknown"))
by_state = (us.groupby("state").agg(net=("Net Order Value", "sum"), items=("Item", "size"),
                                    suppliers=("Supplier", "nunique"))
              .assign(share=lambda d: d.net / d.net.sum()).sort_values("net", ascending=False))
display(by_state.style.format({"net": "{:,.0f}", "share": "{:.1%}"}))

s = by_state.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(s.index, s["net"], color=[MUTED if i == "Unknown" else SERIES[0] for i in s.index], height=0.65)
for y, (v, sh) in enumerate(zip(s["net"], s["share"])):
    ax.text(v, y, f"  ${v / 1e3:,.0f}k · {sh:.0%}", va="center", color=INK_2, fontsize=9)
ax.xaxis.set_major_formatter(usd); ax.set_xlim(0, s["net"].max() * 1.25)
ax.grid(axis="y", visible=False)
ax.set(title="Domestic suppliers: Net PO value by US state (grey = no state in master)",
       xlabel="Net PO value, USD")
save(fig, "q2_by_state")

## Findings for Antonio

**Finding.** **95.4% of Net PO value** goes to domestic (US) suppliers and **4.6% overseas**, mainly Germany (3.2%) and Hungary (0.8%). Measured by items, overseas is much larger (**20.1% of PO items**), so overseas orders are many but small. Among US suppliers, Texas (24%), Nebraska (21%) and Florida (7%) lead, but **45% of domestic value has no state** in the supplier master.

**Decision it supports.** The strategy calls for a globally diversified network. In value terms the network is almost entirely domestic, so if diversification is a goal, overseas sourcing for high-value items is the gap to look at.

**Limitation / extra data.** Country and state are the supplier's *master-data address*, not the ship-from location, and location alone doesn't establish supply risk. 45% of US value can't be placed in a state until the master data is completed. Assessing risk would need lead times, delivery reliability and the actual ship-from location.